In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

data_dir = Path("../data/raw/metadata")

AllClinicalXX – combines all the data from AccelerometryXX,
BiomarkersXX, JointSxXX, MedHistXX, NutritionXX, PhysExamXX, and
SubjectCharXX.

In [ ]:
df = pd.read_csv(data_dir / "AllClinical00.csv")

In [ ]:
print(df.shape)

n_rows, n_cols = df.shape

In [ ]:
df.info()

In [ ]:
prefixes = [col[:3] for col in df.columns]

Counter(prefixes).most_common()

V00 Baseline: Enrollment

P01 Baseline: Screening

P02 Baseline: Initial Eligibility

In [ ]:
for prefix in ['V00', 'P01', 'P02']:
    cols = [c for c in df.columns if c.startswith(prefix)]

    print(f"\n{prefix}: {len(cols)} variables")
    print(cols)

In [ ]:
nutrition = pd.read_csv(data_dir / "oai_nutrition01_definitions.csv")
medical_history = pd.read_csv(data_dir / "oai_oarisk01_definitions.csv")
biomarkers = pd.read_csv(data_dir / "oai_labcollection01_definitions.csv")
comorbidity = pd.read_csv(data_dir / "oai_charlson01_definitions.csv")
physical = pd.read_csv(data_dir / "oai_physfunct01_definitions.csv")
depression = pd.read_csv(data_dir / "oai_ces_d01_definitions.csv")
koos = pd.read_csv(data_dir / "oai_koos_womac01_definitions.csv")
pain = pd.read_csv(data_dir / "oai_oapain01_definitions.csv")
pase = pd.read_csv(data_dir / "oai_pase01_definitions.csv")
sf_12 = pd.read_csv(data_dir / "oai_sf1201_definitions.csv")

In [ ]:
columns = df.columns.tolist()

def remove_prefix(column):
    return re.sub(r'^[A-Z]\d{2}', '', column)

variables = pd.DataFrame({
    "AllClinical00": columns,
    "ElementName_candidate": [remove_prefix(c).lower() for c in columns]
})

print(variables.head())

In [ ]:
def add_dataset_match(df, dictionary, dataset_name):
    mask = df['ElementName_candidate'].isin(dictionary['ElementName'])

    df.loc[mask, 'Dataset'] = dataset_name

    lookup = dictionary.set_index('ElementName')

    mask = df['ElementName_candidate'].isin(lookup.index)

    df.loc[mask, 'Dataset'] = dataset_name

    dictionary_fields = [
        col for col in dictionary.columns
        if col != 'ElementName'
    ]

    for field in dictionary_fields:
        if field not in df.columns:
            df[field] = None

        df.loc[mask, field] = (
            df.loc[mask, 'ElementName_candidate']
            .map(lookup[field])
        )

        
    return df

Koos/Womac data dictionary did not match AllClinical; manual mapping to then replace names

In [ ]:
koos_crosswalk = pd.read_csv(data_dir / "koos_womac_crosswalk.csv")

misc_crosswalk = pd.read_csv(data_dir / "misc_crosswalk.csv")

# Remove white spaces
misc_crosswalk["AllClinical00"] = (
    misc_crosswalk["AllClinical00"].astype("string").str.strip()
)

all_crosswalk = pd.concat(
    [koos_crosswalk, misc_crosswalk],
    ignore_index=True
)


crosswalk_lookup = dict(
    zip(
        all_crosswalk["AllClinical00"],
        all_crosswalk["ElementName"]
    )
)

variables["ElementName_candidate"] = (
    variables["AllClinical00"]
    .map(crosswalk_lookup)
    .fillna(variables["ElementName_candidate"])
)


In [ ]:
variables = add_dataset_match(variables, nutrition, 'Nutrition')
variables = add_dataset_match(variables, medical_history, 'Medical History')
variables = add_dataset_match(variables, biomarkers, 'Biomarkers')
variables = add_dataset_match(variables, comorbidity, 'Comorbidity')
variables = add_dataset_match(variables, physical, 'Physical Function')
variables = add_dataset_match(variables, depression, 'Depression')
variables = add_dataset_match(variables, koos, 'KOOS/WOMAC')
variables = add_dataset_match(variables, pain, 'Pain and Medication')
variables = add_dataset_match(variables, pase, 'Physical Activity')
variables = add_dataset_match(variables, sf_12, 'Medical Outcomes')

In [ ]:
variables['Dataset'].value_counts(dropna=False)

In [ ]:
variables.head()

In [ ]:
manual_info = {
    "ID": {
        "ElementName_candidate": "id",
        "Dataset": "Participant Identifier",
        "ElementDescription": "Release ID",
    },

    "V00HANDED": {
        "ElementName_candidate": "handed",
        "Dataset": "Participant Characteristics",
        "DataType": "Integer",
        "ElementDescription": "Dominant hand for x-ray",
        "ValueRange": "0::3",
        "Notes": "1= Right handed; 2= Left handed; 3= Ambidexterous"
    },
}

for variable, info in manual_info.items():
    mask = variables["AllClinical00"] == variable
    
    for column, value in info.items():
        if column not in variables.columns:
            variables[column] = pd.NA
        
        variables.loc[mask, column] = value

In [ ]:
variables.to_csv(Path("../data/processed") / "AllClinical00_variable_inventory.csv", index=False)

In [ ]:
variables.head()

In [ ]:
df.head(2)

# Check empty in Allclinical

In [ ]:
blank   = df.isna()
coded   = df.astype(str).apply(lambda s: s.str.startswith(".:"))
missing = blank | coded

print("Columns with most missing values:\n",missing.sum().sort_values(ascending=False).head(20) )         # cols with most missing values
print("\nPercentage of missing values per column:",(missing.mean() * 100).round(1))                              # % missing per column
print("\nMissing count per participant:",missing.sum(axis=1))                                          # missing count per participant